<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам конструктора классов с использованием гетторов и сетторов и реализуйте взаимодействие объектов между собой

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

// =====================================================================
// Класс: Товар (Item)
// =====================================================================
public class Item
{
    private int _id;
    private string _name;
    private double _volume;

    public int Id
    {
        get { return _id; }
        set
        {
            if (value <= 0)
                throw new ArgumentException("Идентификатор должен быть положительным числом.");
            _id = value;
        }
    }

    public string Name
    {
        get { return _name; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Название товара не может быть пустым.");
            _name = value;
        }
    }

    public double Volume
    {
        get { return _volume; }
        set
        {
            if (value <= 0)
                throw new ArgumentException("Объем товара должен быть больше 0.");
            _volume = value;
        }
    }

    public Item()
    {
        Id = 1;
        Name = "Неизвестный товар";
        Volume = 0.1;
    }

    public Item(int id, string name, double volume)
    {
        Id = id;
        Name = name;
        Volume = volume;
    }

    public override string ToString()
    {
        return $"[{Id}] {Name} (Объем: {Volume} м³)";
    }
}

// =====================================================================
// Базовый класс: Inventory
// =====================================================================
public class Inventory
{
    private int _warehouseId;
    private string _warehouseName;
    private double _storageCapacity;

    public int WarehouseId
    {
        get { return _warehouseId; }
        set
        {
            if (value <= 0)
                throw new ArgumentException("ID склада должен быть положительным.");
            _warehouseId = value;
        }
    }

    public string WarehouseName
    {
        get { return _warehouseName; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Имя склада не может быть пустым.");
            _warehouseName = value;
        }
    }

    public double StorageCapacity
    {
        get { return _storageCapacity; }
        set
        {
            if (value <= 0)
                throw new ArgumentException("Вместимость склада должна быть больше 0.");
            _storageCapacity = value;
        }
    }

    protected List<Item> Items { get; } = new List<Item>();

    public double UsedCapacity => Items.Sum(i => i.Volume);

    public Inventory()
    {
        WarehouseId = 999;
        WarehouseName = "Стандартный склад";
        StorageCapacity = 100.0;
    }

    public Inventory(int warehouseId, string warehouseName, double storageCapacity)
    {
        WarehouseId = warehouseId;
        WarehouseName = warehouseName;
        StorageCapacity = storageCapacity;
    }

    public virtual string GetStorageStatus()
    {
        double available = StorageCapacity - UsedCapacity;
        return $"Склад '{WarehouseName}' (ID: {WarehouseId}):\n" +
               $"  Занято: {UsedCapacity:F2} из {StorageCapacity:F2} м³ | " +
               $"Свободно: {available:F2} м³ (Товаров: {Items.Count} шт.)";
    }

    public virtual bool AddItem(Item item)
    {
        if (item == null)
        {
            Console.WriteLine("Ошибка: Попытка добавить пустой объект (null).");
            return false;
        }

        if (UsedCapacity + item.Volume > StorageCapacity)
        {
            Console.WriteLine($"[ОТКАЗ] Недостаточно места на складе '{WarehouseName}' для товара {item.Name}.");
            return false;
        }

        Items.Add(item);
        Console.WriteLine($"[УСПЕХ] Товар {item} добавлен на склад '{WarehouseName}'.");
        return true;
    }

    public virtual bool RemoveItem(Item item)
    {
        if (item == null) return false;

        var existingItem = Items.FirstOrDefault(i => i.Id == item.Id);
        if (existingItem != null)
        {
            Items.Remove(existingItem);
            Console.WriteLine($"[УДАЛЕНО] Товар {existingItem.Name} списан со склада '{WarehouseName}'.");
            return true;
        }

        Console.WriteLine($"[ОШИБКА] Товар {item.Name} не найден на складе '{WarehouseName}'.");
        return false;
    }

    // Взаимодействие объектов: перемещение товара с текущего склада на другой
    public bool TransferTo(Inventory destination, Item item)
    {
        Console.WriteLine($"[ПЕРЕМЕЩЕНИЕ] Попытка перевести товар '{item.Name}' со склада '{WarehouseName}' на склад '{destination.WarehouseName}'...");

        var itemOnStock = Items.FirstOrDefault(i => i.Id == item.Id);
        if (itemOnStock == null)
        {
            Console.WriteLine($"[ОШИБКА ПЕРЕМЕЩЕНИЯ] Товар отсутствует на исходном складе '{WarehouseName}'.");
            return false;
        }

        if (destination.UsedCapacity + itemOnStock.Volume > destination.StorageCapacity)
        {
            Console.WriteLine($"[ОШИБКА ПЕРЕМЕЩЕНИЯ] На целевом складе '{destination.WarehouseName}' недостаточно места.");
            return false;
        }

        this.RemoveItem(itemOnStock);
        destination.AddItem(itemOnStock);
        Console.WriteLine($"[ПЕРЕМЕЩЕНИЕ УСПЕШНО] Товар перенесен.\n");
        return true;
    }

    // Взаимодействие объектов: объединение содержимого другого склада в текущий
    public void MergeFrom(Inventory source)
    {
        Console.WriteLine($"[СЛИЯНИЕ] Слияние содержимого склада '{source.WarehouseName}' со складом '{WarehouseName}'...");
        var itemsToMove = new List<Item>(source.Items);

        foreach (var item in itemsToMove)
        {
            if (this.AddItem(item))
            {
                source.RemoveItem(item);
            }
            else
            {
                Console.WriteLine($"[ВНИМАНИЕ] Не удалось переместить {item.Name} из-за нехватки места.");
                break;
            }
        }
    }
}

// =====================================================================
// 1. Производный класс: PersonalInventory
// =====================================================================
public class PersonalInventory : Inventory
{
    private string _ownerName;

    public string OwnerName
    {
        get { return _ownerName; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Имя владельца не может быть пустым.");
            _ownerName = value;
        }
    }

    public PersonalInventory() : base()
    {
        OwnerName = "Частный владелец";
    }

    public PersonalInventory(int warehouseId, string warehouseName, double storageCapacity, string ownerName)
        : base(warehouseId, warehouseName, storageCapacity)
    {
        OwnerName = ownerName;
    }

    public override string GetStorageStatus()
    {
        string baseStatus = base.GetStorageStatus();
        return $"[ПЕРСОНАЛЬНЫЙ СКЛАД Владелец: {OwnerName}]\n{baseStatus}";
    }

    public void ChangeOwner(string newOwnerName)
    {
        Console.WriteLine($"[СМЕНА ВЛАДЕЛЬЦА] Владелец склада '{WarehouseName}' изменен с {OwnerName} на {newOwnerName}.");
        OwnerName = newOwnerName;
    }
}

// =====================================================================
// 2. Производный класс: GroupInventory
// =====================================================================
public class GroupInventory : Inventory
{
    private string _productGroup;

    public string ProductGroup
    {
        get { return _productGroup; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Группа товаров не может быть пустой.");
            _productGroup = value;
        }
    }

    public GroupInventory() : base()
    {
        ProductGroup = "Общая группа";
    }

    public GroupInventory(int warehouseId, string warehouseName, double storageCapacity, string productGroup)
        : base(warehouseId, warehouseName, storageCapacity)
    {
        ProductGroup = productGroup;
    }

    public override bool AddItem(Item item)
    {
        Console.WriteLine($"--> Попытка размещения в секцию категории '{ProductGroup}'...");
        bool isAdded = base.AddItem(item);

        if (isAdded)
        {
            Console.WriteLine($"    [КАТЕГОРИЗАЦИЯ] Товар '{item.Name}' успешно привязан к группе '{ProductGroup}'.");
        }

        return isAdded;
    }

    public void PrintCategorySummary()
    {
        Console.WriteLine($"Групповой склад '{WarehouseName}' специализирован на категории: '{ProductGroup}'.");
    }
}

// =====================================================================
// 3. Производный класс: AutomatedInventory
// =====================================================================
public class AutomatedInventory : Inventory
{
    private string _automationLevel;

    public string AutomationLevel
    {
        get { return _automationLevel; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Уровень автоматизации не может быть пустым.");
            _automationLevel = value;
        }
    }

    public AutomatedInventory() : base()
    {
        AutomationLevel = "Стандартная автоматизация";
    }

    public AutomatedInventory(int warehouseId, string warehouseName, double storageCapacity, string automationLevel)
        : base(warehouseId, warehouseName, storageCapacity)
    {
        AutomationLevel = automationLevel;
    }

    public override bool RemoveItem(Item item)
    {
        Console.WriteLine($"[РОБОТИЗИРОВАННАЯ ВЫГРУЗКА] Инициализация манипулятора (Уровень автоматизации: {AutomationLevel})...");
        bool isRemoved = base.RemoveItem(item);

        if (isRemoved)
        {
            Console.WriteLine($"    Автоматическая система уровня '{AutomationLevel}' завершила извлечение товара.");
        }

        return isRemoved;
    }

    public void RunDiagnostics()
    {
        Console.WriteLine($"[ДИАГНОСТИКА] Роботизированные системы склада '{WarehouseName}' (Уровень: {AutomationLevel}) функционируют нормально.");
    }
}

// =====================================================================
// Исполняемый блок: Демонстрация
// =====================================================================
Console.WriteLine("=====================================================================");
Console.WriteLine("        ДЕМОНСТРАЦИЯ РАБОТЫ ИЕРАРХИИ КЛАССОВ И ВЗАИМОДЕЙСТВИЯ");
Console.WriteLine("=====================================================================\n");

// Создание объектов через конструкторы с параметрами
Item laptop = new Item(101, "Ноутбук Dell", 0.5);
Item chair = new Item(102, "Офисное кресло", 4.0);
Item serverRack = new Item(103, "Серверная стойка", 15.0);

PersonalInventory personalWh = new PersonalInventory(1, "Склад-Бокс А1", 10.0, "Иван Петров");
GroupInventory groupWh = new GroupInventory(2, "Хаб Электроники", 50.0, "Компьютерная техника");
AutomatedInventory autoWh = new AutomatedInventory(3, "Робосклад-Север", 100.0, "Full AI Robotics");

// Наполнение складов
personalWh.AddItem(laptop);
personalWh.AddItem(chair);
groupWh.AddItem(serverRack);

Console.WriteLine("\n--- ВЗАИМОДЕЙСТВИЕ ОБЪЕКТОВ: ПЕРЕМЕЩЕНИЕ ТОВАРА (TransferTo) ---");
// Перемещение товара между складами
personalWh.TransferTo(groupWh, laptop);

Console.WriteLine("--- СТАТУС ПОСЛЕ ПЕРЕМЕЩЕНИЯ ---");
Console.WriteLine(personalWh.GetStorageStatus());
Console.WriteLine();
Console.WriteLine(groupWh.GetStorageStatus());
Console.WriteLine();

Console.WriteLine("--- ВЗАИМОДЕЙСТВИЕ ОБЪЕКТОВ: СЛИЯНИЕ СКЛАДОВ (MergeFrom) ---");
// Робосклад забирает все товары с Хаба Электроники
autoWh.MergeFrom(groupWh);

Console.WriteLine("\n--- ИТОГОВЫЕ СТАТУСЫ ---");
Console.WriteLine(groupWh.GetStorageStatus());
Console.WriteLine(new string('-', 50));
Console.WriteLine(autoWh.GetStorageStatus());

=====================================================================
        ДЕМОНСТРАЦИЯ РАБОТЫ ИЕРАРХИИ КЛАССОВ И ВЗАИМОДЕЙСТВИЯ
=====================================================================

[УСПЕХ] Товар [101] Ноутбук Dell (Объем: 0,5 м³) добавлен на склад 'Склад-Бокс А1'.
[УСПЕХ] Товар [102] Офисное кресло (Объем: 4 м³) добавлен на склад 'Склад-Бокс А1'.
--> Попытка размещения в секцию категории 'Компьютерная техника'...
[УСПЕХ] Товар [103] Серверная стойка (Объем: 15 м³) добавлен на склад 'Хаб Электроники'.
    [КАТЕГОРИЗАЦИЯ] Товар 'Серверная стойка' успешно привязан к группе 'Компьютерная техника'.

--- ВЗАИМОДЕЙСТВИЕ ОБЪЕКТОВ: ПЕРЕМЕЩЕНИЕ ТОВАРА (TransferTo) ---
[ПЕРЕМЕЩЕНИЕ] Попытка перевести товар 'Ноутбук Dell' со склада 'Склад-Бокс А1' на склад 'Хаб Электроники'...
[УДАЛЕНО] Товар Ноутбук Dell списан со склада 'Склад-Бокс А1'.
--> Попытка размещения в секцию категории 'Компьютерная техника'...
[УСПЕХ] Товар [101] Ноутбук Dell (Объем: 0,5 м³) добавлен на склад 'Хаб Электроники'.
    [КАТЕГОРИЗАЦИЯ] Товар 'Ноутбук Dell' успешно привязан к группе 'Компьютерная техника'.
[ПЕРЕМЕЩЕНИЕ УСПЕШНО] Товар перенесен.

--- СТАТУС ПОСЛЕ ПЕРЕМЕЩЕНИЯ ---
[ПЕРСОНАЛЬНЫЙ СКЛАД Владелец: Иван Петров]
Склад 'Склад-Бокс А1' (ID: 1):
  Занято: 4,00 из 10,00 м³ | Свободно: 6,00 м³ (Товаров: 1 шт.)

Склад 'Хаб Электроники' (ID: 2):
  Занято: 15,50 из 50,00 м³ | Свободно: 34,50 м³ (Товаров: 2 шт.)

--- ВЗАИМОДЕЙСТВИЕ ОБЪЕКТОВ: СЛИЯНИЕ СКЛАДОВ (MergeFrom) ---
[СЛИЯНИЕ] Слияние содержимого склада 'Хаб Электроники' со складом 'Робосклад-Север'...
[УСПЕХ] Товар [103] Серверная стойка (Объем: 15 м³) добавлен на склад 'Робосклад-Север'.
[УДАЛЕНО] Товар Серверная стойка списан со склада 'Хаб Электроники'.
[УСПЕХ] Товар [101] Ноутбук Dell (Объем: 0,5 м³) добавлен на склад 'Робосклад-Север'.
[УДАЛЕНО] Товар Ноутбук Dell списан со склада 'Хаб Электроники'.

--- ИТОГОВЫЕ СТАТУСЫ ---
Склад 'Хаб Электроники' (ID: 2):
  Занято: 0,00 из 50,00 м³ | Свободно: 50,00 м³ (Товаров: 0 шт.)
--------------------------------------------------
Склад 'Робосклад-Север' (ID: 3):
  Занято: 15,50 из 100,00 м³ | Свободно: 84,50 м³ (Товаров: 2 шт.)